In [44]:
%%writefile main_turtle.py
#this is for the player turtle

from turtle import Turtle

class MainTurtle(Turtle):
    
    def __init__(self):
        super().__init__()

        self.shape("turtle")
        self.penup()
        self.goto(x = 0, y = -280)
        self.setheading(90)

#set-up up and down movements
    def move_up(self):
        self.forward(distance = 10)
        
    def move_down(self):
        self.backward(distance = 10)

#if the player reaches the other end safely, then some things need to happen
#function to return true if so. will serve as the trigger for subsequent actions
    def is_at_finish_line(self):
        if self.ycor() >= 280:
            return True
        
#if the player reaches the other end safely, then reset to the starting position        
    def go_to_start(self):
        self.goto(x = 0, y = -280)
        
        

Overwriting main_turtle.py


In [45]:
!python main_turtle.py

In [46]:
%%writefile car_manager.py
#class to set up car behaviour


from turtle import Turtle
import random

#list of  colors for the different cars
COLORS = ['red','green','yellow','blue']
MOVING_DISTANCE = 10

class CarManager(Turtle):
    
    def __init__(self):
        super().__init__()
        
#list to append allt the created cars        
        self.all_cars = []
 
    #method to create a car
    def create_car(self):
       #starting position of the cars: at the right edge of the screen 
        x_position = 360
        y_position = random.randint(-250,250) #random positions along y-axis
        
        #setting up the car object
        new_car = Turtle()
        new_car.shape('square') #shape
        new_car.color(random.choice(COLORS))#color
        new_car.shapesize(stretch_len=2)#stretching to 40 pixels wide
        new_car.penup()
        new_car.goto(x_position,y_position)#starting position
        self.all_cars.append(new_car)#appending to the empty list
    
    #moving all the cars added to the list. while the game loop is on, new cars will be
    #continuously created and added to the list.
    def move_car(self):
        for car in self.all_cars:
            car.backward(MOVING_DISTANCE)
            
        

Overwriting car_manager.py


In [47]:
!python car_manager.py

In [65]:
%%writefile scoreboard.py
#to keep track of the score


from turtle import Turtle

class Scoreboard(Turtle):
    def __init__(self):
        super().__init__()
        #defining level attribute. this starts at 1 and gets updated each time the player
        #reaches the other side safely
        self.level = 1 
        self.penup()
        self.hideturtle()
        self.goto(x = -320, y = 260)#position of the scoreboard
    
    #method to display the score
    def display_score(self):
        self.write(f'Level = {self.level}', 
                   move=False, 
                   align="left", 
                   font=("Courier", 18 , "bold"))
    
    #method to update the score
    def level_up(self):
        self.clear() #you need to clear what was there first, otherwise it will just be overwritten
        self.level += 1#adding to the score
        self.display_score()#using the same method to display the score      
   
    #method to trigger the game over sequesnce: move to the center and dipslay GAME OVER message
    def game_over(self):
        self.goto(0,0)
        self.write('GAME OVER', 
                   move=False, 
                   align="center", 
                   font=("Courier", 18 , "bold"))
        

Overwriting scoreboard.py


In [66]:
%%writefile main.py

#this is the main game loop

from turtle import Screen
from main_turtle import MainTurtle
from car_manager import CarManager
from scoreboard import Scoreboard
import time
screen = Screen()
screen.setup(width=800,height=600)
screen.tracer(0)


player = MainTurtle()
car_manager = CarManager()
scoreboard = Scoreboard()
scoreboard.display_score()

#settin up the screen to listen control player movements
screen.listen()
screen.onkey(player.move_up, "Up")
screen.onkey(player.move_down, "Down")


#to control how fast new cars are created. Otherwise, everytime the game loop runs, a new car is
#created

last_car_creation_time = time.time()
car_creation_delay = 0.5

game_on = True

#this is to control the speed of the cars. cars will speed up everytime the the player resets to
#make  it more challenging
initial_speed = 0.25 

while game_on:
    time.sleep(initial_speed)
    screen.update()
    
    current_time = time.time()
    
    #loop to control car creation
    if current_time - last_car_creation_time >= car_creation_delay: 
        car_manager.create_car()
        last_car_creation_time = current_time
    
    #moving the car once created
    car_manager.move_car()
    
    #tracking if the player has collided with a car
    for car in car_manager.all_cars:
        if car.distance(player) < 20: #if the player is less than 20 pixels from the car,
            #then the player will definetly collide.
            game_on = False #changing the flag to stop the game
            scoreboard.game_over() #triggering the game over sequence
    
   #tracking if the player has reached the other end of the screen safely 
    if player.is_at_finish_line():
        player.go_to_start() #go back to the starting posotion
        initial_speed = initial_speed * 0.90 #increase the speed of the cars by 10%
        scoreboard.level_up() #update the scoreboard.
        

screen.exitonclick()

Overwriting main.py


In [67]:
!python main.py